In [ ]:
import math
from mpl_toolkits import mplot3d
import matplotlib.pyplot as plt
import numpy as np
import random

# functions

In [ ]:
def sphere(x):
    """Mínimo global: f(0, ..., 0) = 0"""
    return sum(xi**2 for xi in x)

def rastrigin(x):
    """Mínimo global: f(0, ..., 0) = 0"""
    n = len(x)
    return 10 * n + sum(xi**2 - 10 * np.cos(2 * np.pi * xi) for xi in x)

def rosenbrock(x):
    """Mínimo global: f(1, ..., 1) = 0"""
    return sum(100 * (x[i+1] - x[i]**2)**2 + (x[i] - 1)**2 for i in range(len(x)-1))

# parameters

In [ ]:
DIMENSIONS = 30              # Número de dimensões do espaço de busca
GLOBAL_BEST = 0              # Melhor valor global conhecido
POPULATION = 30              # Número de abelhas no enxame (serão 15 operárias e 15 observadoras)
MAX_ITER = 500               # Número máximo de iterações
LIMIT = 100                  # Limite de tentativas para uma abelha virar escoteira (Scout)
RANDOM_SEED = 42             # Semente aleatória para reprodutibilidade
EXECUTIONS = 30              # Número de execuções para cada função

SELECTION_MODE = 'tournament' # Escolha entre 'roulette' (roleta) ou 'tournament' (torneio)
TOURNAMENT_SIZE = 3          # Quantidade de abelhas no torneio

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# bees

In [ ]:
def get_fitness(value):
    """Calcula a aptidão: menores valores de custo resultam em maior fitness."""
    if value >= 0:
        return 1 / (value + 1)
    else:
        return 1 + abs(value)

def selection(bees, mode='roulette'):
    """Seleção de abelhas operárias pelas observadoras com base em Roleta ou Torneio."""
    if mode == 'roulette':
        fitnesses = [b.fitness for b in bees]
        total_fitness = sum(fitnesses)
        probs = [f / total_fitness for f in fitnesses]
        return np.random.choice(bees, p=probs)
    elif mode == 'tournament':
        competitors = random.sample(bees, TOURNAMENT_SIZE)
        return max(competitors, key=lambda b: b.fitness)

class Bee():
    def __init__(self, b_lo, b_hi, actual_function):
        self.b_lo = b_lo
        self.b_hi = b_hi
        self.actual_function = actual_function
        self.pos = np.random.uniform(b_lo, b_hi, DIMENSIONS)
        self.value = self.actual_function(self.pos)
        self.fitness = get_fitness(self.value)
        self.trials = 0

    def explore(self, target_pos, neighbor_pos):
        """
        Gera uma nova posição variando os dados da equação clássica do ABC:
        v_ij = x_ij + phi * (x_ij - x_kj)
        """
        new_pos = target_pos.copy()
        
        # Modificar apenas uma dimensão (j) aleatoriamente
        j = random.randint(0, DIMENSIONS - 1)
        phi = random.uniform(-1, 1)
        
        new_pos[j] = target_pos[j] + phi * (target_pos[j] - neighbor_pos[j])
        
        # Limita a busca dentro do escopo do problema
        new_pos[j] = max(self.b_lo, min(self.b_hi, new_pos[j]))
        
        new_value = self.actual_function(new_pos)
        new_fitness = get_fitness(new_value)
        
        # Seleção gulosa: se a nova posição for melhor, atualiza.
        if new_fitness > self.fitness:
            self.pos = new_pos
            self.value = new_value
            self.fitness = new_fitness
            self.trials = 0
        else:
            self.trials += 1

    def reset(self):
        """Fase da Escoteira: Reinicializa a abelha aleatoriamente se exceder os limites."""
        self.pos = np.random.uniform(self.b_lo, self.b_hi, DIMENSIONS)
        self.value = self.actual_function(self.pos)
        self.fitness = get_fitness(self.value)
        self.trials = 0

# problem

In [ ]:
def artificial_bee_colony(selection_mode='roulette'):
    resultados_finais = {}

    for actual_function in [sphere, rastrigin, rosenbrock]:
        # Configuração dinâmica de limites para cada função igual no PSO
        if actual_function.__name__ == 'rastrigin':
            b_lo, b_hi = -5.12, 5.12
        elif actual_function.__name__ == 'rosenbrock':
            b_lo, b_hi = -30.0, 30.0
        else: # sphere
            b_lo, b_hi = -100.0, 100.0

        curvas_desta_funcao = []

        for execution in range(EXECUTIONS):
            convergence_curve = []
            print(f"Executando {actual_function.__name__} - Rodada {execution + 1}")
            
            seed_atual = RANDOM_SEED + execution
            random.seed(seed_atual)
            np.random.seed(seed_atual)
            
            # Divide as abelhas em Operárias (Employees) e Observadoras (Onlookers)
            num_employees = POPULATION // 2
            employees = [Bee(b_lo, b_hi, actual_function) for _ in range(num_employees)]
            onlookers = [Bee(b_lo, b_hi, actual_function) for _ in range(POPULATION - num_employees)]
            
            best_value = math.inf
            
            curr_iter = 0
            while curr_iter < MAX_ITER:
                
                # 1. Fase das Abelhas Operárias
                for i, bee in enumerate(employees):
                    k = random.choice([x for x in range(num_employees) if x != i])
                    neighbor_pos = employees[k].pos
                    bee.explore(target_pos=bee.pos, neighbor_pos=neighbor_pos)
                
                # 2. Fase das Abelhas Observadoras
                for bee in onlookers:
                    # Seleciona fonte de acordo com Roleta ou Torneio
                    chosen_employee = selection(employees, mode=selection_mode)
                    
                    k = random.randint(0, num_employees - 1)
                    neighbor_pos = employees[k].pos
                    bee.explore(target_pos=chosen_employee.pos, neighbor_pos=neighbor_pos)
                
                # 3. Fase das Escoteiras (Verifica falhas seguidas e reseta)
                for bee in employees + onlookers:
                    if bee.trials >= LIMIT:
                        bee.reset()
                        
                # 4. Atualiza a melhor solução global encontrada
                current_best = min(employees + onlookers, key=lambda b: b.value).value
                if current_best < best_value:
                    best_value = current_best
                    
                convergence_curve.append(best_value)
                
                # Opcional: Condição de parada (tolerância igual do PSO)
                if abs(best_value - GLOBAL_BEST) < 1e-5:
                    resto_iters = MAX_ITER - curr_iter - 1
                    convergence_curve.extend([best_value] * resto_iters)
                    break
                    
                curr_iter += 1

            curvas_desta_funcao.append(convergence_curve)
        
        media_curvas = np.mean(curvas_desta_funcao, axis=0)
        resultados_finais[actual_function.__name__] = media_curvas

# graph

In [ ]:
# Geração dos gráficos de custo médio para comparação logarítmica
    plt.figure(figsize=(10, 6))
    for nome_funcao, media_curva in resultados_finais.items():
        curva_segura = [max(val, 1e-10) for val in media_curva]
        plt.plot(curva_segura, label=nome_funcao.capitalize(), linewidth=2)

    plt.title(f"Curva de Convergência Média ABC ({EXECUTIONS} Execuções - {DIMENSIONS}D)")
    plt.xlabel("Iterações")
    plt.ylabel("Melhor Custo (Z) - Escala Log")
    plt.yscale("log")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
#rodar
artificial_bee_colony(selection_mode=SELECTION_MODE)